[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Track Objects with BoT-SORT and Re-ID

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/trackers/blob/develop/docs/cookbooks/how-to-add-reid-to-trackers.ipynb)

BoT-SORT matches detections to tracks using box overlap and motion. When a player leaves the frame, BoT-SORT keeps the lost track for a limited time. A player who returns before that track expires may keep the same ID; after it expires, the player gets a new ID.

Re-ID adds appearance: an encoder from the [`reid`](https://github.com/roboflow/re-ID) package turns each player's crop into an embedding. While the lost track is still retained, appearance can help reconnect a returning player even when the boxes no longer overlap. It does not guarantee the match.

This notebook tracks a SportsMOT football clip where the camera pans away from the players and back, with Roboflow's football player detector and BoT-SORT with Re-ID, from the CLI and from Python.

## Setup

### Install dependencies

Dependency conflict warnings are expected and can be ignored.

In [ ]:
!pip install -q "trackers[detection,reid]==2.7.0"

# This vision notebook does not use TorchAudio.
# Remove Colab's preinstalled build to avoid CUDA conflicts after upgrading PyTorch.
%pip uninstall -y torchaudio

Restart the runtime before going on (`Runtime` → `Restart session`): the install upgrades Pillow.

### Check GPU availability

Let's make sure that we have access to a GPU. If no GPU appears, navigate to `Runtime` → `Change runtime type` → `Hardware accelerator`, select a GPU, and click `Save`.

In [ ]:
!nvidia-smi

### Set your Roboflow API key

The player detector loads from Roboflow, so it needs a Roboflow API key. Create a free account at [app.roboflow.com](https://app.roboflow.com), then copy the Private API Key from [Settings → API Keys](https://app.roboflow.com/settings/api). In Colab, open the Secrets panel on the left, add the key as `ROBOFLOW_API_KEY`, and allow this notebook to read it. Running locally, paste the key into the cell below, or export `ROBOFLOW_API_KEY` before starting Jupyter.

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
except ImportError:
    os.environ.setdefault("ROBOFLOW_API_KEY", "YOUR_API_KEY")  # running locally: paste your key here, or export it

### Download the clip

`v_dw7LOz17Omg_c053` from the SportsMOT validation split: 550 frames at 25 FPS.

In [ ]:
from pathlib import Path

SPORTSMOT_VAL = Path("sportsmot/val")
CLIP = "v_dw7LOz17Omg_c053"
FRAMES = SPORTSMOT_VAL / CLIP / "img1"

!trackers download --name sportsmot --split val --asset frames,annotations --output .

### Choose the detector and the tracker settings

We are using the detector `football-players-detection-3zvbc/18` from [Roboflow Universe](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc), which has separate classes for players, goalkeepers and referees. Only players and goalkeepers are tracked.

For Re-ID, we are using the encoder `osnet_x1_0_sportsmot`, an OSNet backbone fine-tuned on SportsMOT.

`REID_FUSION` controls how box and appearance scores combine. `"botsort"`, the default, takes the higher score. `"adaptive"`, from Deep OC-SORT, adds weighted appearance to the box score; we use it for this clip. Both use `reid_config.appearance_threshold` to decide whether the crops look alike enough for appearance to help. Learn more at [Choosing a fusion method](https://trackers.roboflow.com/latest/guides/reid/#choosing-a-fusion-method).

The open proximity gate (`reid_config.proximity_threshold=1.0`) allows appearance to help match a player who returns in a different position while the lost track is still retained. `REID_APPEARANCE_THRESHOLD=0.09` allows appearance to help when the distance between two crops is 0.09 or less; smaller distances mean the crops look more alike. `LOST_TRACK_BUFFER = 150` keeps a lost track for 5 seconds; the default keeps it for about one second. `HIGH_CONF_DET_THRESHOLD` and `MINIMUM_CONSECUTIVE_FRAMES` keep short-lived boxes from becoming IDs. The [Re-ID guide](https://trackers.roboflow.com/latest/guides/reid/) covers every parameter.

In [ ]:
DETECTOR = "football-players-detection-3zvbc/18"
CLASSES = ["player", "goalkeeper"]

REID_MODEL = "osnet_x1_0_sportsmot"

REID_FUSION = "adaptive"
REID_PROXIMITY_THRESHOLD = 1.0
REID_APPEARANCE_THRESHOLD = 0.09
LOST_TRACK_BUFFER = 150
HIGH_CONF_DET_THRESHOLD = 0.7
MINIMUM_CONSECUTIVE_FRAMES = 5
FRAME_RATE = 25

## Track from CLI

`--reid.model` turns Re-ID on. Tracker parameters keep their Python names, with `minimum_` shortened to `min_`.

In [ ]:
TARGET_VIDEO_PATH = f"{CLIP}-reid-cli.mp4"

!trackers track \
    --source {FRAMES} \
    --detection.model {DETECTOR} \
    --detection.confidence 0.2 \
    --filters.classes '[player,goalkeeper]' \
    --tracker botsort \
    --tracker.frame_rate {FRAME_RATE} \
    --tracker.lost_track_buffer {LOST_TRACK_BUFFER} \
    --tracker.high_conf_det_threshold {HIGH_CONF_DET_THRESHOLD} \
    --tracker.min_consecutive_frames {MINIMUM_CONSECUTIVE_FRAMES} \
    --reid.model {REID_MODEL} \
    --tracker.reid_config.fusion {REID_FUSION} \
    --tracker.reid_config.proximity_threshold {REID_PROXIMITY_THRESHOLD} \
    --tracker.reid_config.appearance_threshold {REID_APPEARANCE_THRESHOLD} \
    --output.video {TARGET_VIDEO_PATH} \
    --output.overwrite

### Display result

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH = f"{CLIP}-reid-cli-compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH}

In [ ]:
from IPython.display import Video

Video(TARGET_VIDEO_COMPRESSED_PATH, embed=True, width=1080)

Watch the players on the right around 3 seconds, when the camera pans away. They reappear around 8 seconds: some keep their earlier IDs, while others get new ones. This is partial recovery within the lost-track buffer.

## Track from Python

`AutoModel` loads the same detector as the CLI, and passing `reid_model` turns Re-ID on.

Then pass each frame to `update()`; BoT-SORT crops and embeds the high-confidence detections before matching.

In [ ]:
import numpy as np
import supervision as sv
from inference_models import AutoModel
from reid import ReIDModel

from trackers import BoTSORTTracker, ReIDConfig, frames_from_source

model = AutoModel.from_pretrained(DETECTOR, api_key=os.environ["ROBOFLOW_API_KEY"])
CLASS_IDS = [model.class_names.index(name) for name in CLASSES]
reid_model = ReIDModel.from_pretrained(REID_MODEL)
tracker = BoTSORTTracker(
    reid_model=reid_model,
    reid_config=ReIDConfig(
        fusion=REID_FUSION,
        proximity_threshold=REID_PROXIMITY_THRESHOLD,
        appearance_threshold=REID_APPEARANCE_THRESHOLD,
    ),
    lost_track_buffer=LOST_TRACK_BUFFER,
    high_conf_det_threshold=HIGH_CONF_DET_THRESHOLD,
    minimum_consecutive_frames=MINIMUM_CONSECUTIVE_FRAMES,
    frame_rate=FRAME_RATE,
)

color = sv.ColorPalette.from_hex(
    ["#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
     "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"]
)
box_annotator = sv.BoxAnnotator(color=color, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(
    color=color, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.BLACK, text_scale=0.8
)

TARGET_VIDEO_PATH = f"{CLIP}-reid.mp4"
video_info = sv.VideoInfo(width=1280, height=720, fps=FRAME_RATE)

tracker.reset()
with sv.VideoSink(TARGET_VIDEO_PATH, video_info) as sink:
    for _, frame in frames_from_source(FRAMES):
        detections = model(frame)[0].to_supervision()
        detections = detections[(detections.confidence >= 0.2) & np.isin(detections.class_id, CLASS_IDS)]

        detections = tracker.update(detections, frame=frame)
        detections = detections[detections.tracker_id != -1]

        labels = [f"#{tracker_id}" for tracker_id in detections.tracker_id]
        annotated_frame = box_annotator.annotate(frame.copy(), detections)
        sink.write_frame(label_annotator.annotate(annotated_frame, detections, labels=labels))

### Display result

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH = f"{CLIP}-reid-compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH}

In [ ]:
from IPython.display import Video

Video(TARGET_VIDEO_COMPRESSED_PATH, embed=True, width=1080)

Watch the camera pans. In the first one, players leave the frame on the right at about 3 seconds and reappear around 8 seconds. Most keep their old IDs, while some get new ones. In the later pans around 9 and 14 seconds, the returning players shown keep their IDs. These returns are within the configured lost-track buffer; Re-ID cannot reconnect a track after that buffer expires.

## Tune Re-ID thresholds

How alike should two crops look for appearance to help, and how long should BoT-SORT keep a lost track? The plots below can help you choose `reid_config.appearance_threshold` and `lost_track_buffer` for this clip. The threshold here is 0.09 appearance distance. Smaller distances mean the crops look more alike to the encoder.

In [ ]:
SEQUENCES = [CLIP]

Embed every ground-truth crop. This is the slow cell.

In [ ]:
from trackers.reid import extract_ground_truth_embeddings

embeddings, ids, frame_ids, sequence_ids = extract_ground_truth_embeddings(
    reid_model, SPORTSMOT_VAL, sequences=SEQUENCES, keep_classes=(1,)
)
print(f"{len(embeddings)} crops, {len(np.unique(ids))} identities")

Blue shows pairs of crops from the same player; red shows pairs from different players. For this plot, `maximum_frame_gap` limits the time between crops to the five-second buffer. Pairs farther left look more alike to the encoder.

The dashed line is the notebook's threshold, 0.09. Blue to its left shows same-player pairs that pass the appearance check; red to its left shows different-player pairs that pass too. The printed rates give the share of each. If the colors overlap, moving the line right accepts more same-player pairs but also more possible wrong matches. Passing this check still does not guarantee ID recovery: the lost track must exist and win the match. For how each fusion method uses this threshold, see [Choosing a fusion method](https://trackers.roboflow.com/latest/guides/reid/#choosing-a-fusion-method).

In [ ]:
from math import ceil

import matplotlib.pyplot as plt

from trackers.reid import plot_appearance_distances, sample_appearance_distances

BUFFER_FRAMES = ceil(FRAME_RATE / 30 * LOST_TRACK_BUFFER)
distances = sample_appearance_distances(
    embeddings, ids, frame_ids, sequence_ids,
    same_id_pairs=5000, different_id_pairs=10000,
    minimum_frame_gap=1, maximum_frame_gap=BUFFER_FRAMES,
)
for threshold in (0.05, REID_APPEARANCE_THRESHOLD, 0.15, 0.20):
    same_rate, different_rate = distances.rates_at(threshold)
    print(f"distance {threshold:.3f}  same-person accepted {same_rate:6.1%}  different-person accepted {different_rate:6.1%}")

plot_appearance_distances(distances, thresholds={REID_APPEARANCE_THRESHOLD: "configured threshold"})
plt.show()

The histogram above mixes crop pairs from throughout the five-second buffer. The figure below shows what happens as the gap between crops grows, including beyond that buffer. If the same-player and different-player results stay apart at longer gaps, a longer `lost_track_buffer` may be worth trying. If they come together, keeping tracks longer may not help much. Inspect difficult crops to see which poses or views the encoder struggles with, then check whether its training data covers them.

In the figure below, follow the blue same-player line in the top chart as the gap grows: does it stay below the 0.09 cutoff and apart from the red different-player line? The bottom chart shows how often a same-player pair is closer than a different-player pair (1.0 means always; 0.5 means half the time). For more details, see [How far the threshold carries](https://trackers.roboflow.com/latest/guides/reid/#how-far-the-threshold-carries).

In [ ]:
from trackers.reid import plot_frame_gap_sweep, sweep_frame_gap

sweep = sweep_frame_gap(embeddings, ids, frame_ids, sequence_ids, pairs_per_class=8000)
plot_frame_gap_sweep(sweep, thresholds={REID_APPEARANCE_THRESHOLD: "configured threshold"})
plt.show()

You just used Re-ID to help some players keep their IDs when they returned a few seconds later. Nice work!

The [Re-ID guide](https://trackers.roboflow.com/latest/guides/reid/) covers choosing an encoder, the fusion rules, and the benchmark results.

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/trackers/issues).